### Set Up

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, ttest_ind
from functools import reduce

In [2]:
# Automatically reload modules when they change
%load_ext autoreload
%autoreload 2

In [38]:
scenarios_path = Path().resolve() / "../../scenarios_inputs" / "cheung_variants" 
annotated_output_path = Path().resolve() / "../../annotated_outputs" / "cheung_variants"
plots_path = Path().resolve() / "../../analysis" / "cheung_variants" / "cheung_visualizations"
print(f"scenarios_path: {scenarios_path}")
print(f"annotated_output_path: {annotated_output_path}")

filename_template = "%s_%d_choice_%d.json"


scenarios_path: /Users/anna/Dropbox/2025_moral_scenario_annotation/code/anna/graph_extract/analysis/cheung_variants/../../scenarios_inputs/cheung_variants
annotated_output_path: /Users/anna/Dropbox/2025_moral_scenario_annotation/code/anna/graph_extract/analysis/cheung_variants/../../annotated_outputs/cheung_variants


In [4]:
import analysis_utils

### Select scenario to analyze

In [25]:
#get all of the json files in the annotated_output_path
scenario_name = "bird"
json_files = list(annotated_output_path.glob(f"*{scenario_name}*choice_*.json"))

### Define Functions

In [13]:
#parse the read in filenames to get the condition and id
def parse_filename(filename):
    #filename is of the form "%s_%d_choice_1.json"
    parts = filename.stem.split("_")
    narrative = parts[0]
    id = int(parts[1])
    return narrative, id

In [ ]:
def extract_action(nodes):
    
    value = None

    for node in nodes:
        n = node.get('node',{})
        if n.get('kind') == 'action_choice':
            # print(n.get('label'))
            label = n.get('label')
    
    

    return label

In [53]:
def read_scenario(file_path): 

    narrative, id = parse_filename(file_path)


    with open(file_path, 'r') as f:
        lines = f.readlines()
    
    nodes = [json.loads(line.strip()) for line in lines if line.strip()]

    act_choice = extract_action(nodes)


    # also return original scenario text for reference
    with open(scenarios_path / f"{narrative}.json") as f:

        scenario_list = json.load(f)
        #get the one with the matching id
        this_scenario = next(s for s in scenario_list if s['id'] == id)
     

        #extract the condition from the json read in
        scenario_text = this_scenario['text']
        scenario_deontology = this_scenario['deontology_level']
        scenario_utility = this_scenario['utility_level']
        
    #create a little dictionary with everything in it
    scenario_info = {
        "narrative": narrative,
        "id": id,
        "act": act_choice,
        "text": scenario_text,
        "deontology": scenario_deontology,
        "utility": scenario_utility
    }

    return nodes, scenario_info

In [15]:

def events_to_utility_df(nodes):
  
    util_dfs = []

    for node in nodes:
        n = node.get('node',{})
        if n.get('kind') == 'event':
            links = node.get('links')
            data = {}
            for link in links:
                to_node = link.get('to_node')
                value = link.get('link', {}).get('value')
                data[to_node] = value
                label = n.get('label')
                df = pd.DataFrame([data], index=[label])
            util_dfs.append(df)

    df = pd.concat(util_dfs, ignore_index=False)
    df = df.apply(pd.to_numeric)
    
    return df


In [16]:
def extract_deontology(nodes):
    
    value = None

    for node in nodes:
        n = node.get('node',{})
        if n.get('kind') == 'action_choice':
            # print(n.get('label'))
            links = node.get('links')
            # print(links)
            for link in links:
                    # print(link)
                    if link.get('link', {}).get('kind')=='v-link':
                        value = link.get('link', {}).get('value')
    

    return value

In [17]:
#attempt for a generitc utility function to read the annotation files for a given annotation output json file.
def parse_events_with_labels(file_path):
    """
    Parse a JSON file containing event nodes and extract their C/I/K polarity labels.
    
    Args:
        file_path: Path to the JSON file
        
    Returns:
        dict: Mapping of event labels to their C/I/K polarity strings
              e.g., {"Historical buildings are demolished": "C+I+K+", ...}
    """
    with open(file_path, 'r') as f:
        lines = f.readlines()
    
    # Parse each line to create a list of JSON objects
    nodes = [json.loads(line.strip()) for line in lines if line.strip()]
    
    # Find the first being node (first node with kind "being")
    being_node = None
    for node_obj in nodes:
        if node_obj.get('node', {}).get('kind') == 'being':
            being_node = node_obj
            break
    
    if not being_node:
        return {}
    
    # Create a mapping from event labels to their C/I/K values
    event_labels = {}
    
    for link in being_node.get('links', []):
        to_node_label = link.get('to_node')
        b_link_value = link.get('link', {}).get('value')
        
        if to_node_label and b_link_value:
            event_labels[to_node_label] = b_link_value
    
    # Filter to only include actual events
    event_nodes = {
        node_obj['node']['label'] 
        for node_obj in nodes 
        if node_obj.get('node', {}).get('kind') == 'event'
    }
    
    return {label: value for label, value in event_labels.items() if label in event_nodes}

In [18]:
def get_utilities(nodes):

    util_df =  events_to_utility_df(nodes)


    # For each being (column), calculate product of positive and negative utilities across outcomes
    utility_products = {}
    for col in util_df.columns:
        positive_utils = util_df[col][util_df[col] > 0]
        negative_utils = util_df[col][util_df[col] < 0]
        pos_product = reduce(lambda x, y: x * y, positive_utils, 1) if not positive_utils.empty else 0
        neg_product = reduce(lambda x, y: x * y, negative_utils, 1) if not negative_utils.empty else 0
        utility_products[col] = {"positive_product": pos_product, "negative_product": neg_product, "difference": pos_product - neg_product}

        util_mean_df = pd.DataFrame({'mean_utility': util_df.mean()})
    
    return util_mean_df


### Extract utility and deontology scores from individual scanerios to examine them closely

In [42]:
scenario_dictionary

{'narrative': 'bird',
 'id': 1,
 'text': 'When I was 9 or 10, I had a BB gun I would shoot in our backyard on weekends. One morning, I was shooting at a target. When I was about to shoot, a bird started flying by. I didn’t notice the bird and pulled the trigger as it entered my vision. What happened next seemed like slow motion. The bird fell from flight and I started hearing it frantically wriggling in pain on the ground. I froze for a minute as it slowly started to sink in that I had shot the bird. I frantically ran over. I felt traumatized because I loved animals, especially birds, and I didn’t want to hurt them. I started to cry because I didn’t know what to do next. I knew the bird was in pain and that it would not live. I saw it dying a slow, painful death unless I acted. I had a shovel that I could use to hasten the bird’s death.',
 'deontology': '1',
 'utility': '3'}

In [74]:
scenario_name = "bird"
id = 1
choice = 2
filename = filename_template % (scenario_name, id, choice)
file_path = annotated_output_path / filename
nodes, scenario_dictionary = read_scenario(file_path)
    

do nothing


In [77]:
def single_scenario_analysis(scenario_name, this_id, act_id):

    filename = filename_template % (scenario_name, this_id, act_id)

    file_path = annotated_output_path / filename


    nodes, scenario_dictionary = read_scenario(file_path)
    
    # print(f"Read scenario: {file_path}")

    print(f"Scenario number: {scenario_dictionary['id']}")
    print(f"Scenario action choice: {scenario_dictionary['act']}")
    print(f"Scenario utility: {scenario_dictionary['utility']}")
    print(f"Scenario deontology: {scenario_dictionary['deontology']}")
    util_df =  events_to_utility_df(nodes)


    #get the column means for the utility dataframe (per entity)
    print(util_df.mean())

    util_mean = np.mean(util_df.mean())
    print('overall mean: ' + str(util_mean))

    return util_mean

In [82]:

def get_counterfact_util(scenario_id):

    util_mean_1 = single_scenario_analysis(scenario_name, scenario_id,1)
    print('\n\n')
    util_mean_2 = single_scenario_analysis(scenario_name, scenario_id,2)

    util_total = util_mean_1 -  util_mean_2
    print('\n\nutility difference between choice 1 and choice 2: ' + str(util_total))

    


In [83]:
get_counterfact_util(1)

kill the bird with the shovel
Scenario number: 1
Scenario action choice: kill the bird with the shovel
Scenario utility: 3
Scenario deontology: 1
i          -37.5
the bird    -5.0
dtype: float64
overall mean: -21.25



do nothing
Scenario number: 1
Scenario action choice: do nothing
Scenario utility: 3
Scenario deontology: 1
i          -50.00
the bird   -48.75
dtype: float64
overall mean: -49.375


utility difference between choice 1 and choice 2: 28.125


In [84]:
get_counterfact_util(4)

kill the bird with the shovel
Scenario number: 4
Scenario action choice: kill the bird with the shovel
Scenario utility: 2
Scenario deontology: 1
i                  -30.00
the injured bird    -3.25
dtype: float64
overall mean: -16.625



do nothing
Scenario number: 4
Scenario action choice: do nothing
Scenario utility: 2
Scenario deontology: 1
i                  -27.5
the injured bird   -47.5
dtype: float64
overall mean: -37.5


utility difference between choice 1 and choice 2: 20.875


In [85]:
get_counterfact_util(2)

kill the bird with the painless poison
Scenario number: 2
Scenario action choice: kill the bird with the painless poison
Scenario utility: 3
Scenario deontology: 2
i                  -35.00
the injured bird    -6.25
dtype: float64
overall mean: -20.625



do nothing
Scenario number: 2
Scenario action choice: do nothing
Scenario utility: 3
Scenario deontology: 2
i          -40.000000
the bird   -63.333333
dtype: float64
overall mean: -51.66666666666667


utility difference between choice 1 and choice 2: 31.04166666666667


In [86]:
get_counterfact_util(5)

kill the bird with the painless poison
Scenario number: 5
Scenario action choice: kill the bird with the painless poison
Scenario utility: 2
Scenario deontology: 2
i                  -37.5
the injured bird   -32.5
dtype: float64
overall mean: -35.0



do nothing
Scenario number: 5
Scenario action choice: do nothing
Scenario utility: 2
Scenario deontology: 2
i                  -41.25
the injured bird   -53.75
dtype: float64
overall mean: -47.5


utility difference between choice 1 and choice 2: 12.5


In [87]:
get_counterfact_util(3)

call the animal care facility
Scenario number: 3
Scenario action choice: call the animal care facility
Scenario utility: 3
Scenario deontology: 3
i                                                                      10.4
the injured bird                                                       35.0
1+ animal care facility staff members who would respond to the call    -1.0
dtype: float64
overall mean: 14.799999999999999



do nothing
Scenario number: 3
Scenario action choice: do nothing
Scenario utility: 3
Scenario deontology: 3
i          -38.75
the bird   -65.00
dtype: float64
overall mean: -51.875


utility difference between choice 1 and choice 2: 66.675


In [88]:
get_counterfact_util(6)

call the animal care facility
Scenario number: 6
Scenario action choice: call the animal care facility
Scenario utility: 2
Scenario deontology: 3
i         20.0
1 bird    52.5
dtype: float64
overall mean: 36.25



do nothing
Scenario number: 6
Scenario action choice: do nothing
Scenario utility: 2
Scenario deontology: 3
i                  -43.0
the injured bird   -59.6
dtype: float64
overall mean: -51.3


utility difference between choice 1 and choice 2: 87.55


### get aggregate utilities and arrange into dataframe

In [ ]:
rows = []

for file_path in json_files:
    
    nodes, scenario_dictionary = read_scenario(file_path)

    utility_rated = get_utilities(nodes)

    # utility_rated = utility_rated.drop(utility_rated.index[utility_rated.index == 'i'])
    util_mean = float(utility_rated.mean().iloc[0])

    rows.append(
        {
            "scenario_id": scenario_dictionary.get("id"),
            "narrative": scenario_dictionary.get("narrative"),
            "deontology_label": scenario_dictionary.get("deontology"),
            "utility_label": scenario_dictionary.get("utility"),
            "deontology_rating": extract_deontology(nodes),
            "utility_rating": util_mean,
        }
    )

results_df = pd.DataFrame(rows)
results_df.sort_values(by="scenario_id", inplace=True)
results_df

In [ ]:
# Ensure numeric columns are numeric
results_df["deontology_rating"] = pd.to_numeric(results_df["deontology_rating"], errors="coerce")
results_df["utility_rating"] = pd.to_numeric(results_df["utility_rating"], errors="coerce")

# 1) Means/SEs by deontology_label (for both dependent measures)
summary_by_deontology = (
    results_df
    .groupby("deontology_label", dropna=False)
    .agg(
        n=("deontology_rating", "count"),
        deontology_rating_mean=("deontology_rating", "mean"),
        deontology_rating_se=("deontology_rating", "sem"),
        utility_rating_mean=("utility_rating", "mean"),
        utility_rating_se=("utility_rating", "sem"),
    )
    .reset_index()
    .sort_values("deontology_label")
)

# 2) Means/SEs by utility_label (for both dependent measures)
summary_by_utility = (
    results_df
    .groupby("utility_label", dropna=False)
    .agg(
        n=("deontology_rating", "count"),
        deontology_rating_mean=("deontology_rating", "mean"),
        deontology_rating_se=("deontology_rating", "sem"),
        utility_rating_mean=("utility_rating", "mean"),
        utility_rating_se=("utility_rating", "sem"),
    )
    .reset_index()
    .sort_values("utility_label")
)


In [ ]:
summary_by_deontology

In [ ]:
summary_by_utility

In [ ]:

# Long format for plotting both dependent measures together
plot_df = results_df.melt(
    id_vars=["deontology_label", "utility_label"],
    value_vars=["deontology_rating", "utility_rating"],
    var_name="measure",
    value_name="value",
)


In [ ]:
# Ensure label columns are ordered numerically for plotting
deontology_order = sorted(plot_df["deontology_label"].dropna().astype(int).unique())
utility_order = sorted(plot_df["utility_label"].dropna().astype(int).unique())

plot_df["deontology_label"] = pd.Categorical(
    plot_df["deontology_label"].astype(int),
    categories=deontology_order,
    ordered=True
)
plot_df["utility_label"] = pd.Categorical(
    plot_df["utility_label"].astype(int),
    categories=utility_order,
    ordered=True
)

# Re-run the plotting cell after this cell

In [ ]:
analysis_utils.make_deont_util_plot(scenario_name, plot_df, plots_path)